In [1]:
import numpy as np
import h5py as h5
from multiprocessing import Pool



In [1]:
# get total number of cpus available:
# import os
# n_cpus = os.cpu_count()
# print(f"Number of cpus available: {n_cpus}")


Number of cpus available: 64


: 

In [2]:
norm_delta = 100,
norm_vel = 1000,
BoxSize = 25.
grid = 8
grid_sbox = 32
npart_test = 128**3
nMax_h = 20
nvocab = 64
nrand_sel_box = 64
Mstar_cut = 8
subsamp_ds = 1
sdir = '/mnt/home/spandey/ceph/GOTHAM/data/camels'
nsims_offset_all = [0, 250, 500, 750]
nsims_all = [250, 250, 250, 250]

dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed_all, delta_box_all_squeezed_all, params_repeated_all = [], [], []

for ji in range(len(nsims_all)):
    nsims_offset = nsims_offset_all[ji]
    nsims = nsims_all[ji]
    # savefname = f'{sdir}/ALL_data_nspersim_subhalo_density3Dgrid_{grid_sbox}_isim_all_nrandsubsel_{int(nrand_sel_box/subsamp_ds)}_nvocab{nvocab}_lgMmin_{Mstar_cut}.h5'
    savefname = f'{sdir}/SIMS_{nsims_offset}_{nsims_offset+nsims}_data_nspersim_subhalo_density3Dgrid_{grid_sbox}_isim_all_nrandsubsel_{int(nrand_sel_box/subsamp_ds)}_nvocab{nvocab}_lgMmin_{Mstar_cut}.h5'    
    # if rank == 0: print(f"Reading data from {savefname}", flush=True)
    # import time
    # if rank == 0:
    with h5.File(savefname, 'r') as f:
        params_repeated_all_ji = f['params_repeated_all'][()]
        print(params_repeated_all_ji.shape)
        # delta_box_all_squeezed_all = f['delta_box_all_squeezed_all'][()]
        # delta_box_all_squeezed_all = np.moveaxis(delta_box_all_squeezed_all, -1, 1)
        dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed_all_ji = f['dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed_all'][()]
        print(dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed_all_ji.shape)
        nvocab_total = f['nvocab_total'][()]
        grid_size = f['grid'][()]
        start_token = f['start_token'][()]
        pad_token = f['pad_token'][()]
        end_token = f['end_token'][()]
        max_sentence_length = f['max_sentence_length'][()]    
        f.close()
        
    # if rank == 0:
    def read_hdf5_slice(args):
        file_path, dataset_name, slice_index = args
        with h5.File(file_path, 'r') as f:
            dataset = f[dataset_name]
            data_slice = dataset[slice_index, ...]
        return data_slice[None, ...] 

    def load_hdf5_in_parallel(file_path, dataset_name, num_workers=4):
        with h5.File(file_path, 'r') as f:
            dataset = f[dataset_name]
            num_slices = dataset.shape[0]  
            slice_indices = list(range(num_slices))

        args = [(file_path, dataset_name, idx) for idx in slice_indices]

        with Pool(processes=num_workers) as pool:
            slices = pool.map(read_hdf5_slice, args)

        return slices 

    dataset_name = "delta_box_all_squeezed_all"
    # num_workers = 30  
    # count the total number of cpus on the machine
    num_workers = 64
    print(f"Number of workers: {num_workers}")
    slices = load_hdf5_in_parallel(savefname, dataset_name, num_workers)
    combined_matrix = np.concatenate(slices, axis=0)  
    delta_box_all_squeezed_all_ji = np.moveaxis(combined_matrix, -1, 1)

    dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed_all.append(dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed_all_ji)
    delta_box_all_squeezed_all.append(delta_box_all_squeezed_all_ji)
    params_repeated_all.append(params_repeated_all_ji)



(16000, 6)
(16000, 101)
Number of workers: 64
(16000, 6)
(16000, 101)
Number of workers: 64
(16000, 6)
(16000, 101)
Number of workers: 64
(16000, 6)
(16000, 101)
Number of workers: 64


In [3]:
delta_box_all_squeezed_all = np.concatenate(delta_box_all_squeezed_all, axis=0)
dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed_all = np.concatenate(dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed_all, axis=0)
params_repeated_all = np.concatenate(params_repeated_all, axis=0)




In [4]:
delta_box_all_squeezed_all.shape, dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed_all.shape, params_repeated_all.shape



((64000, 30, 32, 32, 32), (64000, 101), (64000, 6))

In [5]:
def get_data_split(dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed, delta_box_all_squeezed, params_all, n1_fac=0.8, n2_fac=1.0):
    n1 = int(n1_fac*len(dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed)) 
    n2 = int(n2_fac*len(dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed)) 
    train_data_halos = dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed[:n1]
    val_data_halos = dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed[n1:n2]

    dm_train = torch.tensor(delta_box_all_squeezed[:n1]).to(torch.float16)
    dm_val = torch.tensor(delta_box_all_squeezed[n1:n2]).to(torch.float16)

    params_all_train = params_all[:n1]
    params_all_val = params_all[n1:n2]

    x = torch.tensor(train_data_halos[:, :-1])
    y = torch.tensor(train_data_halos[:, 1:])
    mask_train_orig = x != 1
    mask_train = torch.logical_not(mask_train_orig)
    masked_logits = torch.zeros(mask_train.shape)
    mask_train_final = masked_logits.masked_fill(mask_train, float('-inf'))
    mask_train = mask_train_final[:,None,:]
    x, y = torch.tensor(x), torch.tensor(y)
    # x_train = x.long()
    # y_train = y.long()

    x_train = x.to(torch.long)
    y_train = y.to(torch.long)

    # dm_train = dm.bfloat16()
    params_all_train = torch.tensor(params_all_train).to(torch.float16)
    mask_train = torch.tensor(mask_train).to(torch.float16)

    x = torch.tensor(val_data_halos[:, :-1])
    y = torch.tensor(val_data_halos[:, 1:])
    # dm = torch.tensor(val_data_dm)
    mask_val_orig = x != 1
    mask_val = torch.logical_not(mask_val_orig)
    masked_logits = torch.zeros(mask_val.shape)
    mask_val_final = masked_logits.masked_fill(mask_val, float('-inf'))
    mask_val = mask_val_final[:,None,:]
    x, y = torch.tensor(x), torch.tensor(y)
    # x_val = x.long()
    # y_val = y.long()

    x_val = x.to(torch.long)
    y_val = y.to(torch.long)
    # dm_val = dm.bfloat16()
    params_all_val = torch.tensor(params_all_val).to(torch.float16)
    mask_val = torch.tensor(mask_val).to(torch.float16)

    return x_train, y_train, dm_train, params_all_train, mask_train, x_val, y_val, dm_val, params_all_val, mask_val


In [6]:
import torch
x_train, y_train, dm_train, params_train, mask_train, x_val, y_val, dm_val, params_val, mask_val = get_data_split(dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed_all, delta_box_all_squeezed_all, params_repeated_all, 0.8, 1.0)
print(f"Got split with sizes {x_train.shape} and {x_val.shape}", flush=True)        




Got split with sizes torch.Size([51200, 100]) and torch.Size([12800, 100])


/tmp/ipykernel_346436/530219063.py:20: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x, y = torch.tensor(x), torch.tensor(y)
/tmp/ipykernel_346436/530219063.py:29: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  mask_train = torch.tensor(mask_train).to(torch.float16)
/tmp/ipykernel_346436/530219063.py:39: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x, y = torch.tensor(x), torch.tensor(y)
/tmp/ipykernel_346436/530219063.py:47: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTens

In [7]:
int(nrand_sel_box/subsamp_ds)




64

In [8]:
# device_id = rank % torch.cuda.device_count()
Ndevices = 4
sdir = '/mnt/home/spandey/ceph/GOTHAM/data/camels'
savefname = f'{sdir}/SPLIT_DATA_{Ndevices}_gpus_nspersim_subhalo_density3Dgrid_{grid_sbox}_isim_all_nrandsubsel_{int(nrand_sel_box/subsamp_ds)}_nvocab{nvocab}_lgMmin_{Mstar_cut}.h5'
dtype = 'float16'

with h5.File(savefname, 'w') as f:

    for rank in range(Ndevices):

        start = rank * (len(x_train) // Ndevices)
        end = start + (len(x_train) // Ndevices)

        x_train_gpu = (x_train[start:end,...])
        dm_train_gpu = (dm_train[start:end,...])
        params_train_gpu = (params_train[start:end,...])
        mask_train_gpu = (mask_train[start:end,...])
        y_train_gpu = (y_train[start:end,...])
        print(x_train_gpu.shape, dm_train_gpu.shape, params_train_gpu.shape, mask_train_gpu.shape, y_train_gpu.shape)
        # print(f"I am rank {rank} and will process train data from {start} to {end}.")
        # if rank == 0: print(f"Transferred train data to GPU", flush=True)        
        f.create_dataset(f'x_train_dev_{rank}', data=x_train_gpu)
        f.create_dataset(f'dm_train_dev_{rank}', data=dm_train_gpu, dtype=dtype)
        f.create_dataset(f'params_train_dev_{rank}', data=params_train_gpu)
        f.create_dataset(f'mask_train_dev_{rank}', data=mask_train_gpu)
        f.create_dataset(f'y_train_dev_{rank}', data=y_train_gpu)


        start = rank * (len(x_val) // Ndevices)
        end = start + (len(x_val) // Ndevices)
        x_val_gpu = (x_val[start:end,...])
        dm_val_gpu = (dm_val[start:end,...])
        params_val_gpu = (params_val[start:end,...])
        mask_val_gpu = (mask_val[start:end,...])
        y_val_gpu = (y_val[start:end,...])
        print(x_val_gpu.shape, dm_val_gpu.shape, params_val_gpu.shape, mask_val_gpu.shape, y_val_gpu.shape)

        f.create_dataset(f'x_val_dev_{rank}', data=x_val_gpu)
        f.create_dataset(f'dm_val_dev_{rank}', data=dm_val_gpu, dtype=dtype)
        f.create_dataset(f'params_val_dev_{rank}', data=params_val_gpu)
        f.create_dataset(f'mask_val_dev_{rank}', data=mask_val_gpu)
        f.create_dataset(f'y_val_dev_{rank}', data=y_val_gpu)

        # print(f"I am rank {rank} and will process val data from {start} to {end}.")    
        # if rank == 0: print(f"Transferred test data to GPU", flush=True)        

    # nvocab_total = f['nvocab_total'][()]
    # grid_size = f['grid'][()]
    # start_token = f['start_token'][()]
    # pad_token = f['pad_token'][()]
    # end_token = f['end_token'][()]
    # max_sentence_length = f['max_sentence_length'][()]    
    f.create_dataset('nvocab_total', data=nvocab_total)
    f.create_dataset('grid', data=grid_size)
    f.create_dataset('start_token', data=start_token)
    f.create_dataset('pad_token', data=pad_token)
    f.create_dataset('end_token', data=end_token)
    f.create_dataset('max_sentence_length', data=max_sentence_length)

f.close()


torch.Size([12800, 100]) torch.Size([12800, 30, 32, 32, 32]) torch.Size([12800, 6]) torch.Size([12800, 1, 100]) torch.Size([12800, 100])
torch.Size([3200, 100]) torch.Size([3200, 30, 32, 32, 32]) torch.Size([3200, 6]) torch.Size([3200, 1, 100]) torch.Size([3200, 100])
torch.Size([12800, 100]) torch.Size([12800, 30, 32, 32, 32]) torch.Size([12800, 6]) torch.Size([12800, 1, 100]) torch.Size([12800, 100])
torch.Size([3200, 100]) torch.Size([3200, 30, 32, 32, 32]) torch.Size([3200, 6]) torch.Size([3200, 1, 100]) torch.Size([3200, 100])
torch.Size([12800, 100]) torch.Size([12800, 30, 32, 32, 32]) torch.Size([12800, 6]) torch.Size([12800, 1, 100]) torch.Size([12800, 100])
torch.Size([3200, 100]) torch.Size([3200, 30, 32, 32, 32]) torch.Size([3200, 6]) torch.Size([3200, 1, 100]) torch.Size([3200, 100])
torch.Size([12800, 100]) torch.Size([12800, 30, 32, 32, 32]) torch.Size([12800, 6]) torch.Size([12800, 1, 100]) torch.Size([12800, 100])
torch.Size([3200, 100]) torch.Size([3200, 30, 32, 32, 32

In [13]:
# torch.bfloat16




torch.bfloat16